In [1]:
import torch
from torch import nn

In [2]:
!pip install tensorboard

In [3]:
from torch.utils.tensorboard import SummaryWriter

In [4]:
writer = SummaryWriter()

In [5]:
import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int=NUM_WORKERS
):
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  class_names = train_data.classes

  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True,
  )
  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=True,
  )

  return train_dataloader, test_dataloader, class_names

In [6]:
class DesertClassifier(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)  # default stride value is same as kernel_size
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 16 * 16,
                      out_features=output_shape)
        )

    def forward(self, x: torch.Tensor):
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))

In [9]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer):

    model.train()

    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):

        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc


def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):
    model.eval()

    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            test_pred_logits = model(X)

            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5,
          experiment_name = "experiment"):
    
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
               }

    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer)
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn)

        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

        writer = SummaryWriter(log_dir = f"run/{experiment_name}")
        
        writer.add_scalars(main_tag = "Loss",
                           tag_scalar_dict={
                               "train_loss" : train_loss,
                               "test_loss" : test_loss
                           },
                           global_step = epoch)

        writer.add_scalars(main_tag="Accuracy",
                           tag_scalar_dict={
                               "train_acc" : train_acc,
                               "test_acc" : test_acc
                           },
                           global_step = epoch)

        writer.add_graph(model = model,
                        input_to_model = torch.rand(32, 3, 64, 64)
                        )

    writer.close()
        
    return results

In [10]:
    NUM_EPOCHS = 3
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001

    train_dir = "data/desert101/train"
    test_dir = "data/desert101/test"

    data_transform = transforms.Compose(
        [
            transforms.Resize((64, 64)),
            transforms.ToTensor()
        ]
    )

    train_dataloader, test_dataloader, class_names = create_dataloaders(
        train_dir=train_dir,
        test_dir=test_dir,
        transform=data_transform,
        batch_size=BATCH_SIZE
    )

    model1 = DesertClassifier(
        input_shape=3,
        hidden_units=10,
        output_shape=len(class_names)
    )

    model2 = DesertClassifier(
        input_shape=3,
        hidden_units=32,
        output_shape=len(class_names)
    )

In [11]:
loss_fn = torch.nn.CrossEntropyLoss()

In [12]:
train(model=model1,
      train_dataloader=train_dataloader,
      test_dataloader=test_dataloader,
      loss_fn=loss_fn,
      optimizer=torch.optim.Adam(model1.parameters(), lr=LEARNING_RATE),
      epochs=NUM_EPOCHS,
      experiment_name="hidden_units_10"
     )

C:\Users\SERKAN\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch: 1 | train_loss: 1.3903 | train_acc: 0.2455 | test_loss: 1.3887 | test_acc: 0.1979
Epoch: 2 | train_loss: 1.3868 | train_acc: 0.2241 | test_loss: 1.3841 | test_acc: 0.2292
Epoch: 3 | train_loss: 1.3832 | train_acc: 0.2701 | test_loss: 1.3719 | test_acc: 0.4439


{'train_loss': [1.3902968883514404, 1.3868258118629455, 1.3831849217414856],
 'train_acc': [0.24553571428571427, 0.22410714285714284, 0.2700892857142857],
 'test_loss': [1.3887454668680828, 1.384122610092163, 1.3718795379002888],
 'test_acc': [0.19791666666666666, 0.22916666666666666, 0.4439102564102564]}

In [13]:
train(model=model2,
      train_dataloader=train_dataloader,
      test_dataloader=test_dataloader,
      loss_fn=torch.nn.CrossEntropyLoss(),
      optimizer=torch.optim.Adam(model1.parameters(), lr=LEARNING_RATE),
      epochs=NUM_EPOCHS,
      experiment_name="hidden_units_32"
     )

C:\Users\SERKAN\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch: 1 | train_loss: 1.3867 | train_acc: 0.2228 | test_loss: 1.3919 | test_acc: 0.1562
Epoch: 2 | train_loss: 1.3866 | train_acc: 0.2232 | test_loss: 1.3919 | test_acc: 0.1562
Epoch: 3 | train_loss: 1.3867 | train_acc: 0.2214 | test_loss: 1.3919 | test_acc: 0.1562


{'train_loss': [1.386655592918396, 1.3866411209106446, 1.3867401361465455],
 'train_acc': [0.22276785714285716, 0.22321428571428573, 0.22142857142857145],
 'test_loss': [1.391926646232605, 1.391926646232605, 1.391926646232605],
 'test_acc': [0.15625, 0.15625, 0.15625]}

In [16]:
%load_ext tensorboard
%tensorboard --logdir run

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
